# Generic Feature Engineering Pipeline

**Responsibility of this notebook (and nothing more):**

```
RAW COMPANY DATA -> STANDARDIZATION -> VALIDATION -> GENERIC FEATURE ENGINEERING -> CLEAN MODELING DATASET
```

A company only supplies **raw** business data (date, product, price, units sold, optional cost / stock / competitor price / discount / promotion ...).
Every engineered feature (time, price, demand-history, inventory, promotion) is derived here.

This notebook does **not** train models, estimate elasticity or optimise prices.

| # | Section |
|---|---------|
| 1 | Imports |
| 2 | Configuration |
| 3 | Load Raw Data |
| 4 | Column Standardization |
| 5 | Data Type Standardization |
| 6 | Data Validation, Demand Target Handling & Cleaning |
| 7 | Time Features |
| 8 | Price Features |
| 9 | Historical Demand Features |
| 10 | Inventory Features |
| 11 | Promotion Features |
| 12 | Final Feature Engineering Function |
| 13 | Leakage & Feature Safety Checks |
| 14 | Missing Values |
| 15 | Final Dataset |
| 16 | Save Output |
| 17 | Summary |

> Note on order: data types are standardized (section 5) *before* validation (section 6), because checks such as `Price >= 0` are only meaningful on numeric columns. Validation still only **reports**; rows are only removed in the separate cleaning step.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

## 2. Configuration

Everything company/dataset-specific lives in this one cell.

* The `*_COLUMN` values map **canonical names** (left) to the **company's raw column names** (right). Set a value to `None` when the company does not have that column.
* In the final application these mappings will be supplied by the **Streamlit column-mapping interface**; here they are hard-set to the current demo dataset (`Data/demand_forecasting.csv`).
* Nothing else in the notebook refers to a specific dataset, product, store or category.

In [2]:
# ---------------- Raw data location ----------------
DATA_PATH = "../Data/demand_forecasting.csv"      # demo dataset (will come from the Streamlit upload later)
OUTPUT_PATH = None                                # None -> "<input name>_engineered.csv" next to the input file
DATE_FORMAT = None                                # e.g. "%d/%m/%Y" - set it when dates are written day-first / ambiguous;
                                                  # None lets pandas infer the format

# ---------------- Column mapping: company column name -> canonical name ----------------
# Required
DATE_COLUMN             = "Date"
PRODUCT_ID_COLUMN       = "Product ID"
PRICE_COLUMN            = "Price"
DEMAND_COLUMN           = "Demand"          # a company may map "Units Sold" here (see section 6 for the caveat)

# Optional (use None when the company does not have the column)
STORE_ID_COLUMN         = "Store ID"
UNIT_COST_COLUMN        = None              # not present in the demo dataset
INVENTORY_COLUMN        = "Inventory Level"
COMPETITOR_PRICE_COLUMN = "Competitor Pricing"
DISCOUNT_COLUMN         = "Discount"        # expected as a percentage, 0-100
PROMOTION_COLUMN        = "Promotion"       # 0/1 (yes/no style text is also accepted)
CATEGORY_COLUMN         = "Category"
REGION_COLUMN           = "Region"
WEATHER_COLUMN          = "Weather Condition"
SEASONALITY_COLUMN      = "Seasonality"
EPIDEMIC_COLUMN         = "Epidemic"

COLUMN_MAPPING = {
    "Date": DATE_COLUMN,
    "Product_ID": PRODUCT_ID_COLUMN,
    "Store_ID": STORE_ID_COLUMN,
    "Price": PRICE_COLUMN,
    "Demand": DEMAND_COLUMN,
    "Unit_Cost": UNIT_COST_COLUMN,
    "Inventory": INVENTORY_COLUMN,
    "Competitor_Price": COMPETITOR_PRICE_COLUMN,
    "Discount": DISCOUNT_COLUMN,
    "Promotion": PROMOTION_COLUMN,
    "Category": CATEGORY_COLUMN,
    "Region": REGION_COLUMN,
    "Weather": WEATHER_COLUMN,
    "Seasonality": SEASONALITY_COLUMN,
    "Epidemic": EPIDEMIC_COLUMN,
}

# ---------------- Pipeline settings ----------------
HISTORY_WINDOW = 7            # number of previous observations used for the rolling demand average
MAX_DROP_FRACTION = 0.05      # cleaning stops with an error if it would remove more than this share of rows
KEEP_UNMAPPED_COLUMNS = True  # keep columns the company did not map (e.g. "Units Sold") in the output file;
                              # they are carried along as-is and are NEVER treated as model features
SAVE_WITH_ORIGINAL_COLUMN_NAMES = True   # output file uses the company's own column names so the existing
                                         # Modeling notebook keeps working; set False for canonical names
CHECK_LEGACY_COMPATIBILITY = True        # demo dataset only: verify the columns the current project expects

## 3. Load Raw Data

In [3]:
def resolve_path(path):
    """Return an existing Path. Tolerates Data/ vs data/ casing differences on case-sensitive file systems."""
    p = Path(path)
    if p.exists():
        return p
    parts = list(p.parts)
    for i, part in enumerate(parts[:-1]):
        for alt in {part.lower(), part.capitalize(), part.upper()}:
            cand = Path(*parts[:i], alt, *parts[i + 1:])
            if cand.exists():
                return cand
    raise FileNotFoundError(f"Could not find data file: {path}")


data_file = resolve_path(DATA_PATH)
output_file = Path(OUTPUT_PATH) if OUTPUT_PATH else data_file.with_name(data_file.stem + "_engineered.csv")

raw_df = pd.read_csv(data_file)
print(f"Loaded {data_file}  ->  {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns")
raw_df.head()

Loaded ..\Data\demand_forecasting.csv  ->  76,000 rows x 16 columns


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


## 4. Column Standardization

`standardize_columns(df, column_mapping)` renames company-specific columns to the **canonical schema**:

`Date, Product_ID, Price, Demand` (required) and
`Store_ID, Unit_Cost, Inventory, Competitor_Price, Discount, Promotion, Category, Region, Weather, Seasonality, Epidemic` (optional).

It takes an explicit `{canonical_name: company_column}` dictionary. Optional entries that are `None` (or point to a column that does not exist) are skipped with a message; a missing *required* column raises an error. Columns that are not mapped are left untouched.

Example: `{"Date": "date", "Product_ID": "sku", "Price": "selling_price", "Demand": "quantity", "Unit_Cost": "cost", "Inventory": "stock", "Competitor_Price": "competitor_price"}`

In [4]:
CANONICAL_SCHEMA = {
    "Date": "required", "Product_ID": "required", "Price": "required", "Demand": "required",
    "Store_ID": "optional", "Unit_Cost": "optional", "Inventory": "optional",
    "Competitor_Price": "optional", "Discount": "optional", "Promotion": "optional",
    "Category": "optional", "Region": "optional", "Weather": "optional",
    "Seasonality": "optional", "Epidemic": "optional",
}
REQUIRED_COLUMNS = [c for c, kind in CANONICAL_SCHEMA.items() if kind == "required"]
NUMERIC_COLUMNS = ["Price", "Demand", "Unit_Cost", "Inventory", "Competitor_Price", "Discount"]
FLAG_COLUMNS = ["Promotion", "Epidemic"]                                  # 0/1 indicators
TEXT_COLUMNS = ["Product_ID", "Store_ID", "Category", "Region", "Weather", "Seasonality"]


def standardize_columns(df, column_mapping, verbose=True):
    """Rename company-specific columns to canonical names using an explicit mapping dictionary."""
    unknown = [k for k in column_mapping if k not in CANONICAL_SCHEMA]
    if unknown:
        raise ValueError(f"Unknown canonical names in mapping: {unknown}. Valid names: {list(CANONICAL_SCHEMA)}")

    mapping = {k: v for k, v in column_mapping.items() if v}              # None / "" -> not provided
    missing_required = [c for c in REQUIRED_COLUMNS if c not in mapping]
    if missing_required:
        raise ValueError(f"No source column mapped for required field(s): {missing_required}")

    sources = list(mapping.values())
    duplicated = sorted({s for s in sources if sources.count(s) > 1})
    if duplicated:
        raise ValueError(f"The same source column is mapped to several canonical names: {duplicated}")

    absent = {k: v for k, v in mapping.items() if v not in df.columns}
    absent_required = {k: v for k, v in absent.items() if CANONICAL_SCHEMA[k] == "required"}
    if absent_required:
        raise KeyError(f"Required mapped column(s) not found in the data: {absent_required}")
    for canon, src in absent.items():
        if verbose:
            print(f"  [skip] optional column for {canon} ('{src}') not found in data")
        mapping.pop(canon)

    clashes = [c for c in df.columns if c in mapping and mapping[c] != c]
    if clashes:
        raise ValueError(f"Columns already named like canonical fields but mapped elsewhere: {clashes}")

    out = df.rename(columns={src: canon for canon, src in mapping.items()}).copy()
    if verbose:
        print("Column mapping applied (company -> canonical):")
        for canon, src in mapping.items():
            print(f"  {src!r:>24} -> {canon}")
        kept = [c for c in out.columns if c not in CANONICAL_SCHEMA]
        if kept:
            print(f"Unmapped columns left untouched: {kept}")
    return out


df_std = standardize_columns(raw_df, COLUMN_MAPPING)
UNMAPPED_COLUMNS = [c for c in df_std.columns if c not in CANONICAL_SCHEMA]
df_std.head()

Column mapping applied (company -> canonical):
                    'Date' -> Date
              'Product ID' -> Product_ID
                'Store ID' -> Store_ID
                   'Price' -> Price
                  'Demand' -> Demand
         'Inventory Level' -> Inventory
      'Competitor Pricing' -> Competitor_Price
                'Discount' -> Discount
               'Promotion' -> Promotion
                'Category' -> Category
                  'Region' -> Region
       'Weather Condition' -> Weather
             'Seasonality' -> Seasonality
                'Epidemic' -> Epidemic
Unmapped columns left untouched: ['Units Sold', 'Units Ordered']


,Date,Store_ID,Product_ID,Category,Region,Inventory,Units Sold,Units Ordered,Price,Discount,Weather,Promotion,Competitor_Price,Seasonality,Epidemic,Demand
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59


## 5. Data Type Standardization

`standardize_dtypes(df)` converts `Date` to datetime, business numbers to numeric, 0/1 indicators to numeric (yes/no text is understood), and identifiers/categories to string.

Values that cannot be converted become `NaN`/`NaT` - **nothing is dropped here**. The function reports how many invalid values it found so the next sections can deal with them explicitly.

In [5]:
def _id_to_string(s):
    """IDs -> string. Integer-like floats (1.0) become '1', blanks become missing."""
    if pd.api.types.is_float_dtype(s):
        non_null = s.dropna()
        if len(non_null) and (non_null == non_null.round()).all():
            s = s.astype("Int64")
    out = s.astype("string").str.strip()
    return out.mask(out == "", pd.NA)


def _to_numeric(s):
    if pd.api.types.is_bool_dtype(s):
        return s.astype(float)
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype("string").str.strip()
        s = s.mask(s == "", pd.NA)
    out = pd.to_numeric(s, errors="coerce").astype("float64")
    return out.replace([np.inf, -np.inf], np.nan)


_FLAG_TEXT = {"1": 1, "true": 1, "yes": 1, "y": 1, "t": 1, "0": 0, "false": 0, "no": 0, "n": 0, "f": 0}


def _to_flag(s):
    if pd.api.types.is_bool_dtype(s) or pd.api.types.is_numeric_dtype(s):
        return _to_numeric(s)
    text = s.astype("string").str.strip().str.lower()
    mapped = text.map(lambda v: _FLAG_TEXT.get(v, np.nan) if pd.notna(v) else np.nan)
    return pd.to_numeric(mapped, errors="coerce").astype("float64")


def standardize_dtypes(df, verbose=True, date_format=None):
    """Standardize column dtypes. Returns (dataframe, report). Invalid values -> NaN/NaT, never dropped.
    Ambiguous day/month dates (05/03/2023) should be disambiguated with date_format, e.g. "%d/%m/%Y"."""
    out = df.copy()
    rows = []

    def record(col, before_dtype, original, converted):
        invalid = int((original.notna() & converted.isna()).sum())
        rows.append({"column": col, "dtype_before": str(before_dtype), "dtype_after": str(converted.dtype),
                     "missing_in_raw": int(original.isna().sum()), "invalid_set_to_missing": invalid})

    if "Date" in out.columns:
        original = out["Date"]
        converted = pd.to_datetime(original, errors="coerce", format=date_format)
        record("Date", original.dtype, original, converted)
        out["Date"] = converted

    for col in NUMERIC_COLUMNS:
        if col in out.columns:
            original = out[col]
            converted = _to_numeric(original)
            record(col, original.dtype, original, converted)
            out[col] = converted

    for col in FLAG_COLUMNS:
        if col in out.columns:
            original = out[col]
            converted = _to_flag(original)
            record(col, original.dtype, original, converted)
            if converted.notna().all() and converted.isin([0, 1]).all():
                converted = converted.astype("int64")
            out[col] = converted

    for col in TEXT_COLUMNS:
        if col in out.columns:
            original = out[col]
            converted = _id_to_string(original)
            record(col, original.dtype, original, converted)
            out[col] = converted

    report = pd.DataFrame(rows).set_index("column")
    if verbose:
        print("Data type standardization report:")
        print(report.to_string())
        total_invalid = int(report["invalid_set_to_missing"].sum())
        if total_invalid:
            print(f"\n{total_invalid:,} invalid value(s) were set to missing (NOT dropped). See cleaning step.")
        else:
            print("\nNo invalid values found while converting types.")
    return out, report


df_typed, dtype_report = standardize_dtypes(df_std, date_format=DATE_FORMAT)

Data type standardization report:
                 dtype_before     dtype_after  missing_in_raw  invalid_set_to_missing
column                                                                               
Date                   object  datetime64[ns]               0                       0
Price                 float64         float64               0                       0
Demand                  int64         float64               0                       0
Inventory               int64         float64               0                       0
Competitor_Price      float64         float64               0                       0
Discount                int64         float64               0                       0
Promotion               int64         float64               0                       0
Epidemic                int64         float64               0                       0
Product_ID             object          string               0                       0
Store_ID            

## 6. Data Validation, Demand Target Handling & Cleaning

### 6a. Validation (reports only)
`validate_raw_data(df)` checks required columns, non-negative price/demand/cost/inventory, valid dates, missing product IDs, duplicates, missing values, number of products/stores and the date range. **It never modifies or drops data.**

In [6]:
def validate_raw_data(df, verbose=True):
    """Report data-quality problems without changing the data."""
    errors, warns, stats = [], [], {}

    missing_req = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing_req:
        errors.append(f"Required columns missing: {missing_req}")

    def flag(n, message):
        if n:
            warns.append(f"{n:,} {message}")

    stats["rows"] = len(df)
    if "Date" in df.columns:
        flag(int(df["Date"].isna().sum()), "rows with missing/invalid Date")
        valid_dates = df["Date"].dropna()
        stats["date_min"] = valid_dates.min() if len(valid_dates) else None
        stats["date_max"] = valid_dates.max() if len(valid_dates) else None
    if "Product_ID" in df.columns:
        flag(int(df["Product_ID"].isna().sum()), "rows with missing Product_ID")
        stats["n_products"] = int(df["Product_ID"].nunique())
    if "Price" in df.columns:
        flag(int(df["Price"].isna().sum()), "rows with missing/invalid Price")
        flag(int((df["Price"] < 0).sum()), "rows with negative Price")
    if "Demand" in df.columns:
        flag(int(df["Demand"].isna().sum()), "rows with missing/invalid Demand")
        flag(int((df["Demand"] < 0).sum()), "rows with negative Demand")
    if "Unit_Cost" in df.columns:
        flag(int((df["Unit_Cost"] < 0).sum()), "rows with negative Unit_Cost")
    if "Inventory" in df.columns:
        flag(int((df["Inventory"] < 0).sum()), "rows with negative Inventory")
    if "Discount" in df.columns:
        flag(int(((df["Discount"] < 0) | (df["Discount"] > 100)).sum()),
             "rows with Discount outside 0-100 (Discount is expected as a percentage)")
    if "Store_ID" in df.columns:
        stats["n_stores"] = int(df["Store_ID"].nunique())

    stats["exact_duplicate_rows"] = int(df.duplicated().sum())
    flag(stats["exact_duplicate_rows"], "exact duplicate rows")
    key = [c for c in ["Date", "Store_ID", "Product_ID"] if c in df.columns]
    if len(key) >= 2:
        stats["duplicate_keys"] = int(df.duplicated(subset=key).sum())
        if stats["duplicate_keys"] > stats["exact_duplicate_rows"]:
            warns.append(f"{stats['duplicate_keys'] - stats['exact_duplicate_rows']:,} additional rows share the same "
                         f"{'+'.join(key)} with different values (kept; check your data)")

    miss = df.isna().sum()
    miss = miss[miss > 0]
    stats["missing_summary"] = pd.DataFrame({"missing": miss, "pct": (miss / max(len(df), 1) * 100).round(2)}) \
        .sort_values("missing", ascending=False)

    result = {"passed": not errors, "errors": errors, "warnings": warns, "stats": stats}
    if verbose:
        print("=" * 60, "\nRAW DATA VALIDATION REPORT\n" + "=" * 60)
        print(f"Rows: {stats['rows']:,}")
        if stats.get("date_min") is not None:
            print(f"Date range: {stats['date_min'].date()} -> {stats['date_max'].date()}")
        print(f"Unique products: {stats.get('n_products', 'n/a')}")
        print(f"Unique stores: {stats['n_stores']}" if "n_stores" in stats else "Unique stores: n/a (no Store_ID column)")
        print("\nMissing values:")
        print(stats["missing_summary"].to_string() if len(miss) else "  none")
        print("\nProblems found:" if (errors or warns) else "\nNo problems found.")
        for e in errors:
            print("  [ERROR]  ", e)
        for w in warns:
            print("  [WARNING]", w)
    return result


validation = validate_raw_data(df_typed)
assert validation["passed"], validation["errors"]

RAW DATA VALIDATION REPORT
Rows: 76,000
Date range: 2022-01-01 -> 2024-01-30
Unique products: 20
Unique stores: 5

Missing values:
  none

No problems found.


### 6b. Demand target handling - important limitation

The canonical target is **`Demand`**. If a company only has **Units Sold**, it can be mapped to `Demand`, but:

> **Units Sold is only a reasonable proxy for demand when stockouts (censored demand) are not a major issue.**
> When a product sells out, units sold is capped by stock and understates what customers wanted.

When `Inventory` is available, the helper below flags *potential* stockout situations. It is a **diagnostic only**: it uses same-row demand, so it is deliberately *not* added as a feature (that would leak the target).

In [7]:
def flag_potential_stockouts(df, verbose=True):
    """Diagnostic: how often could observed demand be censored by stock? Returns counts (adds no columns)."""
    if "Inventory" not in df.columns:
        if verbose:
            print("Stockout check skipped because Inventory column is unavailable. "
                  "Demand is used as provided - treat it as true demand only if stockouts are rare.")
        return None
    valid = df["Inventory"].notna() & df["Demand"].notna()
    zero_stock = int(((df["Inventory"] == 0) & valid).sum())
    at_or_above = int(((df["Demand"] >= df["Inventory"]) & valid).sum())
    result = {"rows_checked": int(valid.sum()), "zero_inventory_rows": zero_stock,
              "demand_ge_inventory_rows": at_or_above,
              "demand_ge_inventory_pct": round(at_or_above / max(int(valid.sum()), 1) * 100, 2)}
    if verbose:
        print("Potential stockout / censored-demand check (diagnostic only, not a feature):")
        print(f"  rows with Inventory == 0:          {zero_stock:,}")
        print(f"  rows with Demand >= Inventory:     {at_or_above:,} ({result['demand_ge_inventory_pct']}%)")
        if result["demand_ge_inventory_pct"] > 5:
            print("  WARNING: a noticeable share of rows may be stock-constrained; observed demand may understate true demand.")
    return result


stockout_report = flag_potential_stockouts(df_typed)

Potential stockout / censored-demand check (diagnostic only, not a feature):
  rows with Inventory == 0:          406
  rows with Demand >= Inventory:     10,595 (13.94%)


### 6c. Cleaning (separate from validation)

`clean_raw_data(df)` removes only rows that **cannot be used** (missing/invalid Date, Product_ID, Price or Demand, negative Price/Demand, exact duplicates). Impossible *optional* values (negative cost/inventory/competitor price) are set to missing rather than deleting the row. Every removal is logged with its reason, and the step aborts if it would remove more than `MAX_DROP_FRACTION` of the data.

In [8]:
def clean_raw_data(df, max_drop_fraction=0.05, verbose=True):
    """Drop unusable rows (logged by reason). Returns (clean_df, removal_log)."""
    out = df.copy()
    n_start = len(out)
    log = []

    def drop(mask, reason):
        nonlocal out
        n = int(mask.sum())
        if n:
            log.append({"step": "cleaning", "reason": reason, "rows": n})
            out = out.loc[~mask]

    drop(out["Date"].isna(), "Missing/invalid Date")
    drop(out["Product_ID"].isna(), "Missing Product_ID")
    drop(out["Price"].isna(), "Missing/invalid Price")
    drop(out["Price"] < 0, "Negative Price")
    drop(out["Demand"].isna(), "Missing/invalid Demand")
    drop(out["Demand"] < 0, "Negative Demand")
    drop(out.duplicated(), "Exact duplicate row")

    dropped_fraction = (n_start - len(out)) / max(n_start, 1)
    if dropped_fraction > max_drop_fraction:
        raise ValueError(f"Cleaning would remove {dropped_fraction:.1%} of rows (limit {max_drop_fraction:.0%}). "
                         f"Inspect the validation report: {log}")

    out = out.copy()
    nulled = []
    for col in ["Unit_Cost", "Inventory", "Competitor_Price"]:
        if col in out.columns:
            bad = out[col] < 0
            if bad.any():
                out.loc[bad, col] = np.nan
                nulled.append(f"{int(bad.sum()):,} negative {col} value(s) set to missing (rows kept)")

    if verbose:
        print(f"Rows before cleaning: {n_start:,}   after: {len(out):,}   removed: {n_start - len(out):,}")
        for item in log:
            print(f"  removed {item['rows']:,} rows - {item['reason']}")
        for msg in nulled:
            print("  " + msg)
        if not log:
            print("  No rows removed.")
    return out.reset_index(drop=True), log


df_clean, cleaning_log = clean_raw_data(df_typed, max_drop_fraction=MAX_DROP_FRACTION)

Rows before cleaning: 76,000   after: 76,000   removed: 0
  No rows removed.


## 7. Time Features

Derived only from `Date`: `Year, Month, Day, DayOfWeek, WeekOfYear, Quarter, IsWeekend`.

In [9]:
def add_time_features(df):
    """Calendar features from Date (Monday = 0)."""
    dt = df["Date"].dt
    df["Year"] = dt.year.astype(int)
    df["Month"] = dt.month.astype(int)
    df["Day"] = dt.day.astype(int)
    df["DayOfWeek"] = dt.dayofweek.astype(int)
    df["WeekOfYear"] = dt.isocalendar().week.astype(int)
    df["Quarter"] = dt.quarter.astype(int)
    df["IsWeekend"] = (df["DayOfWeek"] >= 5).astype(int)
    return df

## 8. Price Features

| Feature | Definition | Needs |
|---|---|---|
| `Previous_Price` | previous observation's `Price` within the Store+Product (or Product) series | - |
| `Price_Change` | `Price - Previous_Price` | - |
| `Price_Change_Pct` | `Price_Change / Previous_Price * 100` (undefined if previous price is 0) | - |
| `Effective_Price` | `Price * (1 - Discount/100)`; a missing/absent Discount is treated as 0% | Discount optional |
| `Price_Difference` | `Price - Competitor_Price` | Competitor_Price |
| `Price_Ratio` | `Price / Competitor_Price` (undefined if competitor price is 0) | Competitor_Price |

Price change only looks **backwards** (previous observation) and is computed per series, after chronological sorting.

In [10]:
def add_price_features(df, group_keys, skipped):
    """Price features. `df` must already be sorted by group_keys + Date."""
    grouped_price = df.groupby(group_keys, sort=False, dropna=False)["Price"]
    df["Previous_Price"] = grouped_price.shift(1)
    df["Price_Change"] = df["Price"] - df["Previous_Price"]
    df["Price_Change_Pct"] = df["Price_Change"] / df["Previous_Price"].where(df["Previous_Price"] > 0) * 100

    discount = df["Discount"].fillna(0) if "Discount" in df.columns else 0.0   # no discount info -> assume 0%
    df["Effective_Price"] = df["Price"] * (1 - discount / 100)

    if "Competitor_Price" in df.columns:
        competitor = df["Competitor_Price"]
        df["Price_Difference"] = df["Price"] - competitor
        df["Price_Ratio"] = df["Price"] / competitor.where(competitor > 0)
    else:
        skipped["Competitor price features"] = "Price_Difference / Price_Ratio skipped because Competitor_Price is unavailable."
    return df

## 9. Historical Demand Features

Per **Store_ID + Product_ID** series (or **Product_ID** only when there is no store), sorted chronologically:

* `Previous_Demand` = `Demand.shift(1)`
* `Previous_7Day_Avg_Demand` = mean of the previous 7 observations = `rolling(7).mean()` applied to the **already shifted** series, so the current row's demand is never included
* `Demand_Trend` = `Previous_Demand - Previous_7Day_Avg_Demand`

Windows are counted in **observations**; `check_series_regularity` reports whether the data is really one row per day, so you know whether "7 observations" means "7 days".
The window length is `HISTORY_WINDOW` (feature name follows it, e.g. `Previous_14Day_Avg_Demand`).

In [11]:
def add_demand_history_features(df, group_keys, window=7):
    """Lag / rolling demand features built only from earlier observations. `df` must be sorted."""
    avg_col = f"Previous_{window}Day_Avg_Demand"
    df["Previous_Demand"] = df.groupby(group_keys, sort=False, dropna=False)["Demand"].shift(1)       # shift BEFORE rolling
    df[avg_col] = (df.groupby(group_keys, sort=False, dropna=False)["Previous_Demand"]
                     .transform(lambda s: s.rolling(window).mean()))
    df["Demand_Trend"] = df["Previous_Demand"] - df[avg_col]
    return df


def check_series_regularity(df, group_keys, verbose=True):
    """Report the spacing between consecutive observations of each series (df must be sorted)."""
    gaps = df.groupby(group_keys, sort=False, dropna=False)["Date"].diff().dt.days.dropna()
    info = {"series": int(df.groupby(group_keys, sort=False, dropna=False).ngroups),
            "median_gap_days": float(gaps.median()) if len(gaps) else np.nan,
            "pct_gaps_equal_median": float((gaps == gaps.median()).mean() * 100) if len(gaps) else np.nan,
            "max_gap_days": float(gaps.max()) if len(gaps) else np.nan,
            "same_date_repeats": int((gaps == 0).sum())}
    if verbose:
        print(f"Series (group by {group_keys}): {info['series']:,}; median gap {info['median_gap_days']} day(s); "
              f"{info['pct_gaps_equal_median']:.1f}% of gaps equal the median; longest gap {info['max_gap_days']} day(s); "
              f"repeated dates inside a series: {info['same_date_repeats']:,}")
        if info["pct_gaps_equal_median"] < 90 or info["same_date_repeats"]:
            print("  NOTE: observations are not evenly spaced - 'previous N observations' is not exactly 'previous N days'.")
    return info

## 10. Inventory Features

Only created when `Inventory` exists - nothing is invented otherwise.

* `Inventory_Coverage_Days` = `Inventory / Previous_7Day_Avg_Demand` (how many periods current stock lasts at the recent sales rate; uses past demand only; undefined when recent demand is 0)
* `Low_Inventory` = 1 when current `Inventory` is below the **median of that series' earlier inventory values** (expanding, shifted by one). The previous notebook used one global median over the whole dataset, which also "looks into the future" and ignores that products have different stock scales.

In [12]:
def add_inventory_features(df, group_keys, window, skipped):
    if "Inventory" not in df.columns:
        skipped["Inventory features"] = "Inventory features skipped because Inventory column is unavailable."
        return df
    avg_col = f"Previous_{window}Day_Avg_Demand"
    recent_demand = df[avg_col]
    df["Inventory_Coverage_Days"] = df["Inventory"] / recent_demand.where(recent_demand > 0)

    prior_median = (df.groupby(group_keys, sort=False, dropna=False)["Inventory"]
                      .transform(lambda s: s.shift(1).expanding(min_periods=1).median()))
    low = (df["Inventory"] < prior_median).astype("float64")
    undefined = prior_median.isna() | df["Inventory"].isna()
    low[undefined] = np.nan
    df["Low_Inventory"] = low.astype("int64") if not undefined.any() else low.astype("Int64")
    return df

## 11. Promotion Features

* `Weekend_Promotion` = `IsWeekend * Promotion` (needs `Promotion`)
* `Promotion_Discount` = `Promotion * Discount` (needs `Promotion` and `Discount`)

If the inputs are missing the feature is skipped - no default business data is invented.

In [13]:
def add_promotion_features(df, skipped):
    if "Promotion" not in df.columns:
        skipped["Promotion features"] = "Weekend_Promotion / Promotion_Discount skipped because Promotion is unavailable."
        return df
    df["Weekend_Promotion"] = df["IsWeekend"] * df["Promotion"]
    if "Discount" in df.columns:
        df["Promotion_Discount"] = df["Promotion"] * df["Discount"]
    else:
        skipped["Promotion_Discount"] = "Promotion_Discount skipped because Discount is unavailable."
    return df

## 12. Final Feature Engineering Function

`engineer_features(df, ...)` is the single reusable entry point (the Streamlit app will call it later). It:

1. copies the dataframe, 2. standardizes dtypes, 3. sorts chronologically per series, 4. adds time / price / demand-history / inventory / promotion features, 5. returns the engineered dataframe.

The `*_col` arguments name the columns **in the dataframe you pass in** (default = canonical names). Pass `None` / a non-existent name for optional columns the company does not have. The function does not drop rows; cleaning is a separate step (it raises if `Date`/`Product_ID` are missing so unsafe data cannot slip in).
The returned frame carries a report in `df.attrs["feature_report"]` (created features, skipped feature groups, group keys).

In [14]:
OPTIONAL_PASSTHROUGH = ["Store_ID", "Unit_Cost", "Category", "Region", "Weather", "Seasonality", "Epidemic"]


def engineer_features(df,
                      date_col="Date", product_col="Product_ID", store_col="Store_ID",
                      price_col="Price", demand_col="Demand", unit_cost_col="Unit_Cost",
                      inventory_col="Inventory", competitor_price_col="Competitor_Price",
                      discount_col="Discount", promotion_col="Promotion",
                      history_window=7, verbose=True):
    """Generic feature engineering. Returns an engineered copy (no rows dropped)."""
    roles = {"Date": date_col, "Product_ID": product_col, "Store_ID": store_col, "Price": price_col,
             "Demand": demand_col, "Unit_Cost": unit_cost_col, "Inventory": inventory_col,
             "Competitor_Price": competitor_price_col, "Discount": discount_col, "Promotion": promotion_col}
    present = {canon: actual for canon, actual in roles.items() if actual and actual in df.columns}
    missing_required = [c for c in REQUIRED_COLUMNS if c not in present]
    if missing_required:
        raise KeyError(f"Required column(s) not found in dataframe: {missing_required}")

    # 1-2. copy + standardize (work internally with canonical names; names are restored at the end)
    work = df.rename(columns={actual: canon for canon, actual in present.items()}).copy()
    work, _ = standardize_dtypes(work, verbose=False)
    if work["Date"].isna().any() or work["Product_ID"].isna().any():
        raise ValueError("Date / Product_ID contain missing values - run clean_raw_data() before engineer_features().")

    # 3. chronological order inside every series
    group_keys = ["Store_ID", "Product_ID"] if "Store_ID" in work.columns else ["Product_ID"]
    work = work.sort_values(group_keys + ["Date"], kind="mergesort").reset_index(drop=True)
    input_columns = set(work.columns)
    skipped = {}

    # 4. feature groups
    work = add_time_features(work)
    work = add_price_features(work, group_keys, skipped)
    work = add_demand_history_features(work, group_keys, history_window)
    work = add_inventory_features(work, group_keys, history_window, skipped)
    work = add_promotion_features(work, skipped)

    for col in OPTIONAL_PASSTHROUGH:
        if col not in work.columns:
            skipped[f"Optional column {col}"] = f"{col} not provided - nothing derived from it."

    created = [c for c in work.columns if c not in input_columns]
    work = work.rename(columns={canon: actual for canon, actual in present.items()})
    work.attrs["feature_report"] = {"created": created, "skipped": skipped, "group_keys": group_keys,
                                    "history_window": history_window}
    if verbose:
        print(f"Series grouping: {group_keys}")
        print(f"Created {len(created)} features: {created}")
        if skipped:
            print("Skipped feature groups:")
            for name, reason in skipped.items():
                print(f"  - {name}: {reason}")
    return work


check_series_regularity(df_clean.sort_values(["Store_ID", "Product_ID", "Date"] if "Store_ID" in df_clean.columns
                                             else ["Product_ID", "Date"]),
                        ["Store_ID", "Product_ID"] if "Store_ID" in df_clean.columns else ["Product_ID"])
print()
engineered_df = engineer_features(df_clean, history_window=HISTORY_WINDOW)
FEATURE_REPORT = engineered_df.attrs["feature_report"]
GROUP_KEYS = FEATURE_REPORT["group_keys"]
AVG_COL = f"Previous_{HISTORY_WINDOW}Day_Avg_Demand"
print(f"\nEngineered (pre-modeling) dataset: {engineered_df.shape[0]:,} rows x {engineered_df.shape[1]} columns")
engineered_df.head()

Series (group by ['Store_ID', 'Product_ID']): 100; median gap 1.0 day(s); 100.0% of gaps equal the median; longest gap 1.0 day(s); repeated dates inside a series: 0

Series grouping: ['Store_ID', 'Product_ID']
Created 20 features: ['Year', 'Month', 'Day', 'DayOfWeek', 'WeekOfYear', 'Quarter', 'IsWeekend', 'Previous_Price', 'Price_Change', 'Price_Change_Pct', 'Effective_Price', 'Price_Difference', 'Price_Ratio', 'Previous_Demand', 'Previous_7Day_Avg_Demand', 'Demand_Trend', 'Inventory_Coverage_Days', 'Low_Inventory', 'Weekend_Promotion', 'Promotion_Discount']
Skipped feature groups:
  - Optional column Unit_Cost: Unit_Cost not provided - nothing derived from it.

Engineered (pre-modeling) dataset: 76,000 rows x 36 columns


,Date,Store_ID,Product_ID,Category,Region,Inventory,Units Sold,Units Ordered,Price,Discount,Weather,Promotion,Competitor_Price,Seasonality,Epidemic,Demand,Year,Month,Day,DayOfWeek,WeekOfYear,Quarter,IsWeekend,Previous_Price,Price_Change,Price_Change_Pct,Effective_Price,Price_Difference,Price_Ratio,Previous_Demand,Previous_7Day_Avg_Demand,Demand_Trend,Inventory_Coverage_Days,Low_Inventory,Weekend_Promotion,Promotion_Discount
0,2022-01-01,S001,P0001,Electronics,North,195.0,102,252,72.72,5.0,Snowy,0,85.73,Winter,0,115.0,2022,1,1,5,52,1,1,NaN,NaN,NaN,69.0840,-13.01,0.848244,NaN,NaN,NaN,NaN,<NA>,0,0.0
1,2022-01-02,S001,P0001,Electronics,North,93.0,71,0,65.63,5.0,Snowy,0,73.66,Winter,0,84.0,2022,1,2,6,52,1,1,72.72,-7.09,-9.749725,62.3485,-8.03,0.890986,115.0,NaN,NaN,NaN,1,0,0.0
2,2022-01-03,S001,P0001,Electronics,North,274.0,142,229,68.55,15.0,Snowy,1,80.73,Winter,0,132.0,2022,1,3,0,1,1,0,65.63,2.92,4.449185,58.2675,-12.18,0.849127,84.0,NaN,NaN,NaN,0,0,15.0
3,2022-01-04,S001,P0001,Electronics,North,132.0,42,0,61.66,10.0,Snowy,0,54.88,Winter,0,67.0,2022,1,4,1,1,1,0,68.55,-6.89,-10.051058,55.4940,6.78,1.123542,132.0,NaN,NaN,NaN,1,0,0.0
4,2022-01-05,S001,P0001,Electronics,North,319.0,129,0,59.56,25.0,Snowy,1,57.34,Winter,0,110.0,2022,1,5,2,1,1,0,61.66,-2.10,-3.405774,44.6700,2.22,1.038716,67.0,NaN,NaN,NaN,0,0,25.0


## 13. Leakage & Feature Safety Checks

Features and column roles:

* **Target:** `Demand` (never a feature)
* **Identifiers / time:** `Date`
* **Business reference only:** `Unit_Cost` (kept for later profit calculation, not a demand feature)
* **Unmapped company columns** (e.g. `Units Sold`, `Units Ordered` in the demo data): same-period outcomes, carried along but **never** features
* **Model features:** everything else (raw inputs that exist at decision time + engineered features)

The checks do not just re-run the same pandas call: previous-observation values are rebuilt with a merge, rolling means with cumulative sums, and a **perturbation test** changes one demand value and proves that no feature at or before that row changes.

In [15]:
NON_FEATURE_CANONICAL = ["Date", "Demand", "Unit_Cost"]


def build_feature_list(df):
    """Model feature columns = canonical inputs + engineered columns, excluding target/date/cost/unmapped columns."""
    engineered = set(df.attrs["feature_report"]["created"])
    return [c for c in df.columns
            if c not in NON_FEATURE_CANONICAL and (c in CANONICAL_SCHEMA or c in engineered)]


def _previous_observation(df, group_keys, col):
    """Independent rebuild of 'value of col at the previous observation of the same series' via a merge."""
    tmp = df[group_keys].copy()
    tmp["_pos"] = df.groupby(group_keys, sort=False, dropna=False).cumcount()
    prev = df[group_keys].copy()
    prev["_pos"] = tmp["_pos"] + 1
    prev["_prev"] = df[col].to_numpy()
    merged = tmp.merge(prev, on=group_keys + ["_pos"], how="left")
    return merged["_prev"].to_numpy(dtype=float)


def _same(a, b):
    a = np.asarray(pd.Series(a).astype("float64"))
    b = np.asarray(pd.Series(b).astype("float64"))
    return bool(np.allclose(a, b, rtol=1e-6, atol=1e-8, equal_nan=True))


def run_leakage_checks(df_clean, df_eng, feature_cols, group_keys, window, unmapped_cols=(), verbose=True):
    """Return a list of (check, passed, detail). Raises nothing; caller decides how to react."""
    results = []
    avg_col = f"Previous_{window}Day_Avg_Demand"
    add = lambda name, ok, detail="": results.append((name, bool(ok), detail))

    # 1. target is not a model feature
    add("Target 'Demand' is not in the model feature list", "Demand" not in feature_cols)

    # 2. no feature / carried column is a copy of the target
    suspects = []
    for col in list(feature_cols) + [c for c in unmapped_cols if c in df_eng.columns]:
        s = df_eng[col]
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s) and s.notna().any():
            s = s.astype("float64")
            if _same(s, df_eng["Demand"]):
                suspects.append(f"{col} (identical to Demand)")
            elif s.nunique() > 1 and abs(np.corrcoef(s.fillna(s.mean()), df_eng["Demand"])[0, 1]) > 0.999:
                suspects.append(f"{col} (|corr| > 0.999 with Demand)")
    add("No feature duplicates Demand / Units Sold", not suspects, "; ".join(suspects))

    # 3. date ordering inside every series
    step = df_eng.groupby(group_keys, sort=False, dropna=False)["Date"].diff().dt.days.dropna()
    add("Dates are in chronological order within every series", (step >= 0).all(),
        f"{int((step < 0).sum())} out-of-order steps")

    # 4. Previous_Demand is exactly the previous observation's Demand
    add("Previous_Demand == Demand of the previous observation (current Demand not used)",
        _same(df_eng["Previous_Demand"], _previous_observation(df_eng, group_keys, "Demand")))

    # 5. rolling average == mean of the `window` observations BEFORE the current one (cumulative-sum rebuild)
    pos = df_eng.groupby(group_keys, sort=False, dropna=False).cumcount()
    cs = df_eng.groupby(group_keys, sort=False, dropna=False)["Demand"].cumsum()
    cs_g = lambda k: cs.groupby([df_eng[c] for c in group_keys], sort=False, dropna=False).shift(k)
    expected = (cs_g(1) - cs_g(window + 1).fillna(0)) / window
    expected = expected.where(pos >= window)
    add(f"{avg_col} == mean of the {window} observations before the current one",
        _same(df_eng[avg_col], expected))

    # 6. Previous_Price is also strictly historical
    if "Previous_Price" in df_eng.columns:
        add("Previous_Price == Price of the previous observation",
            _same(df_eng["Previous_Price"], _previous_observation(df_eng, group_keys, "Price")))

    # 7. perturbation test on one real series: changing Demand at row k must not change any
    #    demand-derived feature at rows <= k (no current/future leakage) but must change row k+1 (test is sensitive)
    first_key = df_clean[group_keys].iloc[0]
    mask = np.logical_and.reduce([(df_clean[c] == first_key[c]) for c in group_keys])
    series = df_clean.loc[mask].sort_values(group_keys + ["Date"], kind="mergesort").reset_index(drop=True)
    if len(series) >= window + 3:
        k = len(series) // 2
        perturbed = series.copy()
        perturbed.loc[k, "Demand"] = float(perturbed["Demand"].max()) * 1000 + 1e6
        base_f = engineer_features(series, history_window=window, verbose=False)
        pert_f = engineer_features(perturbed, history_window=window, verbose=False)
        derived = [c for c in ["Previous_Demand", avg_col, "Demand_Trend", "Inventory_Coverage_Days", "Low_Inventory"]
                   if c in base_f.columns]
        unchanged_upto_k = all(_same(base_f.loc[:k, c], pert_f.loc[:k, c]) for c in derived)
        reacts_after_k = not _same(base_f.loc[k + 1, ["Previous_Demand"]], pert_f.loc[k + 1, ["Previous_Demand"]])
        add("Perturbation test: changing Demand at row k leaves all demand features at rows <= k unchanged",
            unchanged_upto_k, f"checked {derived} on one series of {len(series)} rows")
        add("Perturbation test sensitivity: the same change DOES affect the next row (test can detect leakage)",
            reacts_after_k)
    else:
        add("Perturbation test", True, "skipped: first series too short")

    if verbose:
        print("=" * 72, "\nLEAKAGE & FEATURE SAFETY SUMMARY\n" + "=" * 72)
        for name, ok, detail in results:
            print(f"[{'PASS' if ok else 'FAIL'}] {name}" + (f"  -> {detail}" if detail else ""))
        n_ok = sum(ok for _, ok, _ in results)
        print(f"\n{n_ok}/{len(results)} checks passed.")
    return results


FEATURE_COLUMNS = build_feature_list(engineered_df)
leakage_results = run_leakage_checks(df_clean, engineered_df, FEATURE_COLUMNS, GROUP_KEYS, HISTORY_WINDOW,
                                     unmapped_cols=UNMAPPED_COLUMNS)
assert all(ok for _, ok, _ in leakage_results), "Leakage / feature-safety check failed - see summary above"

LEAKAGE & FEATURE SAFETY SUMMARY
[PASS] Target 'Demand' is not in the model feature list
[PASS] No feature duplicates Demand / Units Sold
[PASS] Dates are in chronological order within every series  -> 0 out-of-order steps
[PASS] Previous_Demand == Demand of the previous observation (current Demand not used)
[PASS] Previous_7Day_Avg_Demand == mean of the 7 observations before the current one
[PASS] Previous_Price == Price of the previous observation
[PASS] Perturbation test: changing Demand at row k leaves all demand features at rows <= k unchanged  -> checked ['Previous_Demand', 'Previous_7Day_Avg_Demand', 'Demand_Trend', 'Inventory_Coverage_Days', 'Low_Inventory'] on one series of 760 rows
[PASS] Perturbation test sensitivity: the same change DOES affect the next row (test can detect leakage)

8/8 checks passed.


## 14. Missing Values

Missing lag values at the start of every product/store series are **expected** (there is no earlier observation) and are *not* filled with zeros. They are reported here and handled in the final-dataset step; any other missing values are reported but left as they are.

In [16]:
def history_report(df, window):
    avg_col = f"Previous_{window}Day_Avg_Demand"
    info = {"series": int(df.groupby(GROUP_KEYS, sort=False, dropna=False).ngroups),
            "rows_without_previous_demand": int(df["Previous_Demand"].isna().sum()),
            f"rows_without_{window}_period_history": int(df[avg_col].isna().sum())}
    print("Product/Store history created:")
    print(f"  Number of series (group by {GROUP_KEYS}): {info['series']:,}")
    print(f"  Initial rows without Previous_Demand: {info['rows_without_previous_demand']:,}")
    print(f"  Initial rows without {window}-period history: {info[f'rows_without_{window}_period_history']:,}")
    return info


def missing_summary(df):
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    return pd.DataFrame({"missing": miss, "pct": (miss / max(len(df), 1) * 100).round(2)})


history_info = history_report(engineered_df, HISTORY_WINDOW)
print("\nMissing values in the engineered dataset (before building the modeling dataset):")
ms = missing_summary(engineered_df)
print(ms.to_string() if len(ms) else "  none")

Product/Store history created:
  Number of series (group by ['Store_ID', 'Product_ID']): 100
  Initial rows without Previous_Demand: 100
  Initial rows without 7-period history: 700

Missing values in the engineered dataset (before building the modeling dataset):
                          missing   pct
Demand_Trend                  700  0.92
Inventory_Coverage_Days       700  0.92
Previous_7Day_Avg_Demand      700  0.92
Previous_Price                100  0.13
Previous_Demand               100  0.13
Price_Change_Pct              100  0.13
Price_Change                  100  0.13
Low_Inventory                 100  0.13


## 15. Final Dataset

The **engineered dataset** (all rows, including each series' warm-up rows) stays separate from the **modeling dataset**, which keeps only rows that have the core demand-history features (`Previous_Demand`, `Previous_7Day_Avg_Demand`, `Demand_Trend`). Removed rows and the reason are logged. Inputs are never imputed with fabricated values.

In [17]:
def make_modeling_dataset(df_eng, window, keep_unmapped=True):
    """Keep rows with complete demand-history features. Returns (modeling_df, removal_log)."""
    avg_col = f"Previous_{window}Day_Avg_Demand"
    core = ["Previous_Demand", avg_col, "Demand_Trend"]
    incomplete = df_eng[core].isna().any(axis=1)
    log = []
    if incomplete.any():
        log.append({"step": "modeling dataset", "rows": int(incomplete.sum()),
                    "reason": f"No full history (Previous_Demand / {window}-period average) - "
                              f"first {window} observations of each product/store series"})
    model_df = df_eng.loc[~incomplete].copy()

    created = set(df_eng.attrs["feature_report"]["created"])
    keep = [c for c in df_eng.columns if c in CANONICAL_SCHEMA or c in created or (keep_unmapped and c not in CANONICAL_SCHEMA)]
    model_df = model_df[keep].reset_index(drop=True)
    model_df.attrs["feature_report"] = df_eng.attrs["feature_report"]

    numeric = model_df.select_dtypes(include=[np.number])
    assert not np.isinf(numeric.to_numpy(dtype=float, na_value=np.nan)).any(), "Infinite values found"
    return model_df, log


modeling_df, modeling_log = make_modeling_dataset(engineered_df, HISTORY_WINDOW, keep_unmapped=KEEP_UNMAPPED_COLUMNS)
REMOVED_ROWS = cleaning_log + modeling_log

print(f"Engineered dataset: {engineered_df.shape}   ->   Modeling dataset: {modeling_df.shape}")
for item in REMOVED_ROWS:
    print(f"  removed {item['rows']:,} rows ({item['step']}): {item['reason']}")
remaining = missing_summary(modeling_df)
print("\nRemaining missing values in the modeling dataset (kept as-is, not zero-filled):")
print(remaining.to_string() if len(remaining) else "  none")
for col in ["Previous_Demand", AVG_COL, "Demand_Trend"]:
    assert modeling_df[col].notna().all()
modeling_df.head()

Engineered dataset: (76000, 36)   ->   Modeling dataset: (75300, 36)
  removed 700 rows (modeling dataset): No full history (Previous_Demand / 7-period average) - first 7 observations of each product/store series

Remaining missing values in the modeling dataset (kept as-is, not zero-filled):
  none


,Date,Store_ID,Product_ID,Category,Region,Inventory,Units Sold,Units Ordered,Price,Discount,Weather,Promotion,Competitor_Price,Seasonality,Epidemic,Demand,Year,Month,Day,DayOfWeek,WeekOfYear,Quarter,IsWeekend,Previous_Price,Price_Change,Price_Change_Pct,Effective_Price,Price_Difference,Price_Ratio,Previous_Demand,Previous_7Day_Avg_Demand,Demand_Trend,Inventory_Coverage_Days,Low_Inventory,Weekend_Promotion,Promotion_Discount
0,2022-01-08,S001,P0001,Electronics,North,308.0,96,248,68.56,10.0,Snowy,1,80.99,Winter,0,113.0,2022,1,8,5,1,1,1,66.07,2.49,3.768730,61.7040,-12.43,0.846524,87.0,105.857143,-18.857143,2.909582,0,1,10.0
1,2022-01-09,S001,P0001,Electronics,North,212.0,57,0,62.76,10.0,Sunny,0,67.11,Winter,0,87.0,2022,1,9,6,1,1,1,68.56,-5.80,-8.459743,56.4840,-4.35,0.935181,113.0,105.571429,7.428571,2.008119,0,0,0.0
2,2022-01-10,S001,P0001,Electronics,North,403.0,88,0,68.99,5.0,Cloudy,0,81.14,Winter,0,99.0,2022,1,10,0,2,1,0,62.76,6.23,9.926705,65.5405,-12.15,0.850259,87.0,106.000000,-19.000000,3.801887,0,0,0.0
3,2022-01-11,S001,P0001,Electronics,North,315.0,112,0,73.71,0.0,Cloudy,0,85.29,Winter,0,103.0,2022,1,11,1,2,1,0,68.99,4.72,6.841571,73.7100,-11.58,0.864228,99.0,101.285714,-2.285714,3.110014,0,0,0.0
4,2022-01-12,S001,P0001,Electronics,North,203.0,70,179,65.76,10.0,Snowy,0,60.13,Winter,0,85.0,2022,1,12,2,2,1,0,73.71,-7.95,-10.785511,59.1840,5.63,1.093630,103.0,106.428571,-3.428571,1.907383,1,0,0.0


## 16. Save Output

By default the file keeps the company's **original column names** (inverse of the mapping) so the existing `Modeling.ipynb` keeps working; engineered columns have the same names as before (`Price_Difference`, `Previous_Demand`, ...). Set `SAVE_WITH_ORIGINAL_COLUMN_NAMES = False` to save canonical names instead.

In [18]:
output_df = modeling_df.copy()
if SAVE_WITH_ORIGINAL_COLUMN_NAMES:
    inverse_mapping = {canon: company for canon, company in COLUMN_MAPPING.items()
                       if company and canon in output_df.columns}
    output_df = output_df.rename(columns=inverse_mapping)

output_df.to_csv(output_file, index=False)
print(f"Saved {output_file}  ({output_df.shape[0]:,} rows x {output_df.shape[1]} columns)")

# read back to verify the file
check = pd.read_csv(output_file, nrows=5)
assert list(check.columns) == list(output_df.columns)
assert sum(1 for _ in open(output_file, encoding="utf-8")) - 1 == len(output_df)
print("Round-trip check passed (columns and row count match).")

if CHECK_LEGACY_COMPATIBILITY:
    LEGACY_FEATURES = [
        "Price", "Discount", "Promotion", "Competitor Pricing", "Price_Difference", "Price_Ratio",
        "Effective_Price", "Promotion_Discount", "Price_Change", "Price_Change_Pct", "Previous_Demand",
        "Previous_7Day_Avg_Demand", "Demand_Trend", "Inventory Level", "Inventory_Coverage_Days", "Category",
        "Region", "Weather Condition", "Seasonality", "Epidemic", "Year", "Month", "Day", "DayOfWeek",
        "IsWeekend", "Weekend_Promotion", "Low_Inventory", "Store ID", "Product ID", "Demand", "Date",
    ]
    absent = [c for c in LEGACY_FEATURES if c not in check.columns]
    print("\nCurrent-project compatibility:",
          "all columns used by the existing project are present." if not absent else f"MISSING {absent}")
    assert not absent

Saved ..\Data\demand_forecasting_engineered.csv  (75,300 rows x 36 columns)
Round-trip check passed (columns and row count match).

Current-project compatibility: all columns used by the existing project are present.


## 17. Summary

In [19]:
FINAL_FEATURES = build_feature_list(modeling_df)
engineered_names = FEATURE_REPORT["created"]
carried = [c for c in modeling_df.columns if c not in FINAL_FEATURES and c not in ("Date", "Demand")]

print("=" * 72, "\nFEATURE ENGINEERING SUMMARY\n" + "=" * 72)
print(f"Raw shape:         {raw_df.shape}")
print(f"Engineered shape:  {engineered_df.shape}   (all rows, incl. series warm-up rows)")
print(f"Modeling shape:    {modeling_df.shape}   (saved to {output_file})")
print(f"Date range:        {modeling_df['Date'].min().date()} -> {modeling_df['Date'].max().date()}")
print(f"Products:          {modeling_df['Product_ID'].nunique():,}")
print(f"Stores:            {modeling_df['Store_ID'].nunique():,}" if "Store_ID" in modeling_df.columns
      else "Stores:            n/a (no Store_ID column)")
print(f"Series grouping:   {GROUP_KEYS}")

total_removed = sum(item["rows"] for item in REMOVED_ROWS)
print(f"\nRows removed in total: {total_removed:,}")
for item in REMOVED_ROWS:
    print(f"  - {item['rows']:,} rows ({item['step']}): {item['reason']}")
if not REMOVED_ROWS:
    print("  - none")

print("\nMissing values in final modeling dataset:")
print(remaining.to_string() if len(remaining) else "  none")

print(f"\nFinal feature list ({len(FINAL_FEATURES)} features):")
for i, name in enumerate(FINAL_FEATURES, 1):
    tag = "engineered" if name in engineered_names else "input"
    print(f"  {i:2d}. {name}  [{tag}]")
print(f"\nTarget: Demand      Date column: Date")
print(f"Carried through, NOT features: {carried}")

if FEATURE_REPORT["skipped"]:
    print("\nSkipped feature groups:")
    for name, reason in FEATURE_REPORT["skipped"].items():
        print(f"  - {name}: {reason}")

print("\nLeakage & feature safety checks:")
for name, ok, detail in leakage_results:
    print(f"  [{'PASS' if ok else 'FAIL'}] {name}")

FEATURE ENGINEERING SUMMARY
Raw shape:         (76000, 16)
Engineered shape:  (76000, 36)   (all rows, incl. series warm-up rows)
Modeling shape:    (75300, 36)   (saved to ..\Data\demand_forecasting_engineered.csv)
Date range:        2022-01-08 -> 2024-01-30
Products:          20
Stores:            5
Series grouping:   ['Store_ID', 'Product_ID']

Rows removed in total: 700
  - 700 rows (modeling dataset): No full history (Previous_Demand / 7-period average) - first 7 observations of each product/store series

Missing values in final modeling dataset:
  none

Final feature list (32 features):
   1. Store_ID  [input]
   2. Product_ID  [input]
   3. Category  [input]
   4. Region  [input]
   5. Inventory  [input]
   6. Price  [input]
   7. Discount  [input]
   8. Weather  [input]
   9. Promotion  [input]
  10. Competitor_Price  [input]
  11. Seasonality  [input]
  12. Epidemic  [input]
  13. Year  [engineered]
  14. Month  [engineered]
  15. Day  [engineered]
  16. DayOfWeek  [engineered